# Feature Normalization Challenge - Data Exploration

This notebook explores the provided data to understand:
- Taxonomy structure and feature definitions
- Product data format and content
- Ground truth feature labels
- Challenge requirements and constraints

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## 1. Load Taxonomy Data

The taxonomy defines valid features per category

In [2]:
# Load taxonomy
taxonomy = pd.read_parquet('taxonomy/taxonomy.parquet')

print(f"Taxonomy shape: {taxonomy.shape}")
print(f"\nColumns: {taxonomy.columns.tolist()}")
print(f"\nFirst few rows:")
taxonomy.head(10)

Taxonomy shape: (65766, 4)

Columns: ['category', 'feature_name', 'aggregated_feature_values', 'feature_type']

First few rows:


,category,feature_name,aggregated_feature_values,feature_type
0,iso7094,Außen-Ø,"{'[80 mm]','[105 mm]','[44 mm]','[98 mm]','[56 mm]','[34 mm]','[28 mm]','[72 mm]','[22 mm]','[18...",numeric
1,iso7094,Innen-Ø,"{'[13.5 mm]','[11 mm]','[15.5 mm]','[22 mm]','[16 mm]','[9 mm]','[33 mm]','[30 mm]','[5.5 mm]','...",numeric
2,iso7094,Material,"{'[Stahl]','[Edelstahl (A2)]','[Edelstahl (A4)]','[Stahl 10.9]','[Edelstahl (A3)]','[Edelstahl]'}",categorical
3,iso7094,Oberfläche,"{'[galvanisch verzinkt]','[blank]','[feuerverzinkt]','[vernickelt]','[verzinkt gelb]','[brüniert]'}",categorical
4,iso7094,Stärke,"{'[6 mm]','[3 mm]','[4 mm]','[2 mm]','[5 mm]','[8 mm]','[10 mm]'}",numeric
5,iso7094_edelstahl,Material,"{'[A2]','[A4]'}",categorical
6,iso7094_edelstahl,Außen-Ø,"{'[80 mm]','[56 mm]','[44 mm]','[105 mm]','[28 mm]','[22 mm]','[34 mm]','[85 mm]','[72 mm]','[18...",numeric
7,iso7094_edelstahl,Innen-Ø,"{'[50 mm]','[5.5 mm]','[33 mm]','[9 mm]','[13.5 mm]','[11 mm]','[22 mm]','[17.5 mm]','[10 mm]','...",numeric
8,iso7094_edelstahl,Stärke,"{'[2 mm]','[3 mm]','[4 mm]','[5 mm]','[6 mm]'}",numeric
9,iso7379,Länge,"{'[16 mm]','[5 mm]','[20 mm]','[45 mm]','[80 mm]','[50 mm]','[25 mm]','[12 mm]','[10 mm]','[70 m...",numeric


In [3]:
# Taxonomy statistics
print(f"Number of unique categories: {taxonomy['category'].nunique()}")
print(f"Number of unique features: {taxonomy['feature_name'].nunique()}")
print(f"\nFeature types distribution:")
print(taxonomy['feature_type'].value_counts())

print(f"\nCategories:")
print(taxonomy['category'].unique())

Number of unique categories: 23357
Number of unique features: 4178

Feature types distribution:
feature_type
numeric        37504
categorical    28262
Name: count, dtype: int64

Categories:
<ArrowStringArray>
[                     'iso7094',            'iso7094_edelstahl',
                      'iso7379',                      'iso7380',
                      'iso7434',            'iso7434_edelstahl',
                      'iso7435',            'iso7435_edelstahl',
                      'iso8673',                   'iso8673_lg',
 ...
         'tiefentrennschleifer',    'tiefenunterzug_lagerregal',
                    'tiefgrund', 'tiefkaelte_umwaelzthermostat',
            'tiefkuehlaggregat',                 'tiefkuehlbox',
           'tiefkuehletiketten',              'tiefkuehlmarker',
            'tiefkuehlreiniger',               'tiefkuehlzelle']
Length: 23357, dtype: str


In [4]:
# Features per category
features_per_category = taxonomy.groupby('category').size().sort_values(ascending=False)
print("Features per category:")
print(features_per_category)

Features per category:
category
hannspree_monitore                           19
msi_monitore                                 19
philips_monitore                             19
viewsonic_monitore                           19
sharp_nec_monitore_alle_                     19
                                             ..
can_bus_umsetzer                              1
fremdspannungsarme_erde_leiterkennzeichen     1
langschwenkhebel                              1
slotbleche                                    1
froschfingerschiene                           1
Length: 23357, dtype: int64


In [5]:
# Examine aggregated_feature_values for categorical features
categorical_features = taxonomy[taxonomy['feature_type'] == 'categorical']
print(f"\nCategorical features: {len(categorical_features)}")
print("\nSample categorical feature with allowed values:")
sample = categorical_features.iloc[0]
print(f"Category: {sample['category']}")
print(f"Feature: {sample['feature_name']}")
print(f"Allowed values: {sample['aggregated_feature_values']}")


Categorical features: 28262

Sample categorical feature with allowed values:
Category: iso7094
Feature: Material
Allowed values: {'[Stahl]','[Edelstahl (A2)]','[Edelstahl (A4)]','[Stahl 10.9]','[Edelstahl (A3)]','[Edelstahl]'}


## 2. Load Training Data

In [6]:
# Load training products
train_products = pd.read_parquet('train/products.parquet')

print(f"Training products shape: {train_products.shape}")
print(f"\nColumns: {train_products.columns.tolist()}")
print(f"\nFirst few products:")
train_products.head()

Training products shape: (1425067, 4)

Columns: ['uid', 'category', 'title', 'description']

First few products:


,uid,category,title,description
0,B2887-228998_|_sicherheitsascher,sicherheitsascher,"Kombiascher, HxØ 800x310mm, 40l, Korpus Stahl weiß, Löschkopf",Kombination aus Ascher und Abfallbehälter. Der Behälter ist aus beschichtetem Stahlblech. Die be...
1,B7409-2757653-BP_|_modellauto_chassis,modellauto_chassis,Tamiya TT-02 1:10 RC TT-02 Type SRX Chassis Kit 1:10 RC Modellauto,Tamiya TT-02 1:10 RC TT-02 Type SRX Chassis Kit 1:10 RC Modellauto <br><br>1:10 RC TT-02 Type SR...
2,B9113-ROLINE 11045511_|_dvi_kabel,dvi_kabel,"DVI Monitor Kabel DVI 24+1 Stecker, Dual Link, 1 m",ROLINE GOLD DVI Monitorkabel in hochwertiger Qualität mit verbesserter Schirmung für Bildschirmü...
3,B8TFE-4229198_|_dvi_kabel,dvi_kabel,"LINDY DVI-D Kabel DualLink 0,5 m","Gold Line. Produkttyp: Monitorkabel/Adapter, Anschlüsse: DVI-D | DVI-D, Steckertyp: Stecker - St..."
4,B6627-1504101251_|_planmesserkopf,planmesserkopf,Planfräser M5009-063-B22-07-05 WALTER,"Xtra·tec® XT · 8 Schneidkanten pro Wendeschneidplatte · Dc: 63,0mm · d1: 22,0mm · l4: 40,0mm · L..."


In [7]:
# Training products statistics
print(f"Number of training products: {len(train_products)}")
print(f"\nProducts per category:")
print(train_products['category'].value_counts())

print(f"\nTitle length statistics:")
train_products['title_length'] = train_products['title'].str.len()
print(train_products['title_length'].describe())

print(f"\nDescription length statistics:")
train_products['desc_length'] = train_products['description'].str.len()
print(train_products['desc_length'].describe())

Number of training products: 1425067

Products per category:
category
it_servicevertraege                      56809
system_arbeitsplatz                      42347
patchpanel                               25277
geradverschraubung                       25019
uebergangsstueck                         23188
                                         ...  
laufsohlenplatte                             1
symantec_security_suite_lizenzen             1
verbotszeichen_din_4844_2_d_p030             1
gewinde_alu_bogen                            1
federverschlussglied_mit_winkellasche        1
Name: count, Length: 3240, dtype: int64[pyarrow]

Title length statistics:
count    1425067.0
mean     57.619969
std      23.046545
min            2.0
25%           40.0
50%           52.0
75%           72.0
max          180.0
Name: title_length, dtype: Float64

Description length statistics:
count    1.425067e+06
mean     6.301860e+02
std      7.274946e+02
min      0.000000e+00
25%      1.150000e+02
50%      

In [8]:
# Sample product examples
print("Sample product examples:\n")
for idx in range(min(3, len(train_products))):
    product = train_products.iloc[idx]
    print(f"Product {idx+1}:")
    print(f"  UID: {product['uid']}")
    print(f"  Category: {product['category']}")
    print(f"  Title: {product['title'][:150]}...")
    print(f"  Description: {product['description'][:200]}...")
    print()

Sample product examples:

Product 1:
  UID: B2887-228998_|_sicherheitsascher
  Category: sicherheitsascher
  Title: Kombiascher, HxØ 800x310mm, 40l, Korpus Stahl weiß, Löschkopf...
  Description: Kombination aus Ascher und Abfallbehälter. Der Behälter ist aus beschichtetem Stahlblech. Die besondere Deckelkonstruktion verhindert einen permanenten Sauerstoffaustausch, so dass Feuer und Glut gelö...

Product 2:
  UID: B7409-2757653-BP_|_modellauto_chassis
  Category: modellauto_chassis
  Title: Tamiya TT-02 1:10 RC TT-02 Type SRX Chassis Kit 1:10 RC Modellauto...
  Description: Tamiya TT-02 1:10 RC TT-02 Type SRX Chassis Kit 1:10 RC Modellauto <br><br>1:10 RC TT-02 Type SRX Chassis Kit<br><br>Bausatzmodell im Maßstab 1/10<br>4WD Antrieb über Kardanwelle, Einzelradaufhängung ...

Product 3:
  UID: B9113-ROLINE 11045511_|_dvi_kabel
  Category: dvi_kabel
  Title: DVI Monitor Kabel DVI 24+1 Stecker, Dual Link, 1 m...
  Description: ROLINE GOLD DVI Monitorkabel in hochwertiger Qualität mit ver

## 3. Load Training Ground Truth Features

In [9]:
# Load training features (ground truth)
train_features = pd.read_parquet('train/product_features.parquet')

print(f"Training features shape: {train_features.shape}")
print(f"\nColumns: {train_features.columns.tolist()}")
print(f"\nFirst few features:")
train_features.head(10)

Training features shape: (3221974, 4)

Columns: ['uid', 'feature_name', 'feature_value', 'feature_type']

First few features:


,uid,feature_name,feature_value,feature_type
0,B2887-228998_|_sicherheitsascher,Material,Stahl,categorical
1,B7409-2757653-BP_|_modellauto_chassis,Maßstab,:10,numeric
2,B9113-ROLINE 11045511_|_dvi_kabel,Pinbelegung,DVI-D 24+1 (Dual-Link),categorical
3,B9113-ROLINE 11045511_|_dvi_kabel,Länge,1 m,numeric
4,B8TFE-4229198_|_dvi_kabel,Pinbelegung,DVI-D 24+1 (Dual-Link),categorical
5,B8TFE-4229198_|_dvi_kabel,Länge,0.5 m,numeric
6,B6627-1504101251_|_planmesserkopf,Schneiden-Ø,22 mm,numeric
7,B6627-1504101251_|_planmesserkopf,Schneidenanzahl,7-schneidig,numeric
8,BD3XQ-1786235_|_dorn_presseinsatz,Querschnitt,10 mm²,numeric
9,B9323-6023665_|_kaffeemaschinen_isolierkanne,Volumen,1 l,numeric


In [10]:
# Feature statistics
print(f"Total feature instances: {len(train_features)}")
print(f"Unique products with features: {train_features['uid'].nunique()}")
print(f"\nFeatures per product:")
features_per_product = train_features.groupby('uid').size()
print(features_per_product.describe())

print(f"\nFeature type distribution:")
print(train_features['feature_type'].value_counts())

Total feature instances: 3221974
Unique products with features: 1425067

Features per product:
count    1.425067e+06
mean     2.260928e+00
std      1.392712e+00
min      1.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      3.000000e+00
max      1.100000e+01
dtype: float64

Feature type distribution:
feature_type
numeric        1774768
categorical    1447206
Name: count, dtype: int64[pyarrow]


In [11]:
# Example: Show all features for one product
sample_uid = train_features['uid'].iloc[0]
sample_product = train_products[train_products['uid'] == sample_uid].iloc[0]
sample_features = train_features[train_features['uid'] == sample_uid]

print(f"Sample Product Analysis:")
print(f"\nUID: {sample_uid}")
print(f"Category: {sample_product['category']}")
print(f"Title: {sample_product['title']}")
print(f"Description: {sample_product['description'][:300]}...")
print(f"\nExtracted Features ({len(sample_features)}):")
print(sample_features[['feature_name', 'feature_value', 'feature_type']])

Sample Product Analysis:

UID: B2887-228998_|_sicherheitsascher
Category: sicherheitsascher
Title: Kombiascher, HxØ 800x310mm, 40l, Korpus Stahl weiß, Löschkopf
Description: Kombination aus Ascher und Abfallbehälter. Der Behälter ist aus beschichtetem Stahlblech. Die besondere Deckelkonstruktion verhindert einen permanenten Sauerstoffaustausch, so dass Feuer und Glut gelöscht werden. Deckel und Boden schwarz matt beschichtet. Einwurf mit Kantenschutz.<br>...

Extracted Features (4):
        feature_name feature_value feature_type
0           Material         Stahl  categorical
2590539     Breite/Ø        310 mm      numeric
2590540        Farbe          weiß  categorical
2590541      Volumen          40 l      numeric


## 4. Load Validation Data

In [12]:
# Load validation products
val_products = pd.read_parquet('val/products.parquet')
val_features = pd.read_parquet('val/product_features.parquet')

print(f"Validation products: {len(val_products)}")
print(f"Validation features: {len(val_features)}")
print(f"\nValidation products per category:")
print(val_products['category'].value_counts())

Validation products: 405965
Validation features: 1052171

Validation products per category:
category
pressfitting           20128
spanplattenschraube    17311
spreizduebel           14635
hplc_trennsaeule       10401
laboretikett            9495
                       ...  
wandkalender               1
buecher_schule             1
press_t_stueck             1
pancake_objektiv           1
grosstastenhandy           1
Name: count, Length: 1035, dtype: int64[pyarrow]


## 5. Load Test Data

In [13]:
# Load test products
test_products = pd.read_parquet('test/products.parquet')
test_submission = pd.read_parquet('test/submission.parquet')

print(f"Test products: {len(test_products)}")
print(f"Test submission template: {len(test_submission)}")
print(f"\nTest products per category:")
print(test_products['category'].value_counts())

print(f"\nSubmission template columns:")
print(test_submission.columns.tolist())
print(f"\nSubmission template sample:")
test_submission.head()

Test products: 3103603
Test submission template: 7864744

Test products per category:
category
lenkrolle_feststeller          118552
wendeschneidplatten             93394
iso4017                         48383
sechskantschraube_ohne_norm     46365
care_pack                       42246
                                ...  
holzstativ                          1
holzrechen                          1
din9841                             1
schlauchtragekorb                   1
milchdispenser                      1
Name: count, Length: 6573, dtype: int64[pyarrow]

Submission template columns:
['uid', 'feature_name', 'feature_value', 'feature_type']

Submission template sample:


,uid,feature_name,feature_value,feature_type
0,B7299-499409-BP_|_schwingschleifblatt,Ausführung,None,categorical
1,B5585-2170327-BP_|_apparaterolle,Laufbelag,None,categorical
2,PNJMV-828493-MU_|_bit_phillips,Antrieb,None,categorical
3,PNJMV-2611847-MU_|_torx_bit_satz,Anzahl Teile,None,numeric
4,BH9F9-826149_|_magazinschrank,Farbe,None,categorical


## 6. Data Quality Analysis

In [14]:
# Check for missing values
print("Missing values in training products:")
print(train_products.isnull().sum())

print("\nMissing values in training features:")
print(train_features.isnull().sum())

Missing values in training products:
uid             0
category        0
title           0
description     0
title_length    0
desc_length     0
dtype: int64

Missing values in training features:
uid              0
feature_name     0
feature_value    0
feature_type     0
dtype: int64


In [15]:
# Check if all training products have features
products_with_features = set(train_features['uid'].unique())
all_products = set(train_products['uid'].unique())
products_without_features = all_products - products_with_features

print(f"Products with features: {len(products_with_features)}")
print(f"Products without features: {len(products_without_features)}")

Products with features: 1425067
Products without features: 0


## 7. Feature Extraction Patterns Analysis

In [16]:
# Analyze most common features
print("Most common features across all products:")
print(train_features['feature_name'].value_counts().head(20))

Most common features across all products:
feature_name
Länge                 301226
Material              249780
Ausführung            149452
Gewinde-Ø             139807
Durchmesser           128853
Farbe                 116926
Breite                 94263
Oberfläche             82923
Größe                  64986
Garantiezeitraum       56809
Höhe                   50989
Volumen                43259
Form                   43123
Element                42361
Nenn-Ø                 39220
Inhalt                 38727
Tiefe                  37145
Gesamtlänge            35666
Verpackungseinheit     33623
Spannutenlänge         33268
Name: count, dtype: int64[pyarrow]


In [17]:
# Analyze categorical vs numeric features
categorical_count = (train_features['feature_type'] == 'categorical').sum()
numeric_count = (train_features['feature_type'] == 'numeric').sum()

print(f"Categorical feature instances: {categorical_count} ({categorical_count/len(train_features)*100:.1f}%)")
print(f"Numeric feature instances: {numeric_count} ({numeric_count/len(train_features)*100:.1f}%)")

Categorical feature instances: 1447206 (44.9%)
Numeric feature instances: 1774768 (55.1%)


In [18]:
# Sample numeric features to understand format
numeric_features = train_features[train_features['feature_type'] == 'numeric']
print("Sample numeric features:")
print(numeric_features[['feature_name', 'feature_value']].head(20))

Sample numeric features:
       feature_name feature_value
1           Maßstab           :10
3             Länge           1 m
5             Länge         0.5 m
6       Schneiden-Ø         22 mm
7   Schneidenanzahl   7-schneidig
8       Querschnitt        10 mm²
9           Volumen           1 l
10            Größe          PZ 1
12     Klingenlänge        100 mm
13            Größe          PZ 0
15     Klingenlänge        150 mm
16           Inhalt        290 ml
17            Länge         50 mm
18      Durchmesser        3.9 mm
24            Größe           T 8
27            Größe          T 20
29            Größe          T 27
33   Materialstärke         40 µm
34            Länge        350 mm
35           Breite        250 mm


In [19]:
# Sample categorical features to understand values
categorical_features_data = train_features[train_features['feature_type'] == 'categorical']
print("Sample categorical features:")
print(categorical_features_data[['feature_name', 'feature_value']].head(20))

Sample categorical features:
       feature_name                  feature_value
0          Material                          Stahl
2       Pinbelegung         DVI-D 24+1 (Dual-Link)
4       Pinbelegung         DVI-D 24+1 (Dual-Link)
11  Klingenmaterial           Chrom-Vanadium-Stahl
14  Klingenmaterial           Chrom-Vanadium-Stahl
19         Material                 Edelstahl (A4)
20       Oberfläche                          blank
21     Warenzustand                            Neu
22    Schnittstelle                          USB A
23           Bauart                   extern / USB
25  Klingenmaterial           Chrom-Vanadium-Stahl
26  Klingenmaterial  Chrom-Vanadium-Molybdän-Stahl
28  Klingenmaterial           Chrom-Vanadium-Stahl
30    Schnittstelle                          USB C
31           Bauart                   extern / USB
32     Warenzustand                            Neu
39       Oberfläche                          blank
40         Material                 Edelstahl (A2)
42

## 8. Challenge Summary

Based on the data exploration, here's what we need to build:

1. **Input**: Product (uid, category, title, description)
2. **Output**: Features (uid, feature_name, feature_value, feature_type)
3. **Constraints**:
   - Only use features defined in taxonomy for each category
   - Categorical values must match taxonomy's allowed values
   - Numeric values must use correct units/format
4. **Evaluation**: Exact Value Match Accuracy
5. **Optimization**: Balance accuracy (40%), throughput (20%), and cost (40%)

In [20]:
# Summary statistics
print("=" * 60)
print("CHALLENGE DATA SUMMARY")
print("=" * 60)
print(f"\nTaxonomy:")
print(f"  Categories: {taxonomy['category'].nunique()}")
print(f"  Total features defined: {len(taxonomy)}")
print(f"  Categorical features: {(taxonomy['feature_type'] == 'categorical').sum()}")
print(f"  Numeric features: {(taxonomy['feature_type'] == 'numeric').sum()}")

print(f"\nTraining Data:")
print(f"  Products: {len(train_products)}")
print(f"  Feature instances: {len(train_features)}")
print(f"  Avg features per product: {len(train_features)/len(train_products):.2f}")

print(f"\nValidation Data:")
print(f"  Products: {len(val_products)}")
print(f"  Feature instances: {len(val_features)}")

print(f"\nTest Data:")
print(f"  Products: {len(test_products)}")
print(f"  Expected predictions: {len(test_submission)}")
print("=" * 60)

CHALLENGE DATA SUMMARY

Taxonomy:
  Categories: 23357
  Total features defined: 65766
  Categorical features: 28262
  Numeric features: 37504

Training Data:
  Products: 1425067
  Feature instances: 3221974
  Avg features per product: 2.26

Validation Data:
  Products: 405965
  Feature instances: 1052171

Test Data:
  Products: 3103603
  Expected predictions: 7864744
